# Daily ECMWF CF (TIGGE via cdsapi)

Descarga el control forecast (cf) del dataset `tigge-forecasts` en ECMWF Data Stores (`ecds.ecmwf.int`), via `cdsapi`. Credenciales desde Databricks secret scope `ecmwf`. TIGGE tiene latencia de archivo (confirmado empiricamente: ~2 dias); se busca hacia atras desde ese punto. `tp` viene en kg/m2 (equivalente a mm, sin conversion).

In [ ]:
%pip install --quiet cdsapi geopandas pyogrio xarray netCDF4
dbutils.library.restartPython()

In [ ]:
import json
import math
from datetime import date, datetime, timezone
from pathlib import Path

import geopandas as gpd

GRID_DEG = 0.25
GEOJSON_PATH = "/Workspace/Users/joaquintschopp@gmail.com/rio-uruguay-hydro-pipeline/SIG/subcuencas_modelo.geojson"


def compute_download_area(geojson_path=GEOJSON_PATH, grid_deg=GRID_DEG, margin_cells=1):
    gdf = gpd.read_file(geojson_path)
    minx, miny, maxx, maxy = gdf.total_bounds
    margin = grid_deg * margin_cells
    north = math.ceil((maxy + margin) / grid_deg) * grid_deg
    south = math.floor((miny - margin) / grid_deg) * grid_deg
    west = math.floor((minx - margin) / grid_deg) * grid_deg
    east = math.ceil((maxx + margin) / grid_deg) * grid_deg
    return {"north": round(north, 4), "west": round(west, 4), "south": round(south, 4), "east": round(east, 4)}


def area_to_cds_list(area):
    return [area["north"], area["west"], area["south"], area["east"]]


def normalize_longitude(lon):
    return lon - 360 if lon > 180 else lon


def point_in_bbox(lat, lon, area):
    lon_norm = normalize_longitude(lon)
    return (area["south"] <= lat <= area["north"]) and (area["west"] <= lon_norm <= area["east"])


def raw_filename(tipo, run_date, run_time, ext):
    return f"ECMWF_{tipo.upper()}_{run_date:%Y_%m_%d}_t{run_time}.{ext}"


def already_landed(tipo, run_date, run_time, json_dir):
    return (json_dir / raw_filename(tipo, run_date, run_time, "json")).exists()


def _step_to_hours(step_val):
    if step_val is None:
        return 0
    import numpy as np
    if isinstance(step_val, np.timedelta64):
        return int(step_val / np.timedelta64(1, "h"))
    return int(step_val)


def _compute_valid_datetime(run_date, run_time, step_hours):
    from datetime import timedelta
    run_dt = datetime.combine(run_date, datetime.strptime(run_time, "%H").time(), tzinfo=timezone.utc)
    return run_dt + timedelta(hours=step_hours)


def flatten_forecast(ds, run_date, run_time, tipo, source_api, unit_to_mm_factor, area=None, number=None):
    tp = ds["tp"]
    lats = ds["latitude"].values
    lons = ds["longitude"].values

    if "step" in tp.dims:
        steps = ds["step"].values
    else:
        steps = [tp["step"].values] if "step" in tp.coords else [None]
        tp = tp.expand_dims("step") if "step" not in tp.dims else tp

    extracted_at = datetime.now(timezone.utc).isoformat()
    records = []
    for step_idx, step_val in enumerate(steps):
        step_hours = _step_to_hours(step_val)
        valid_dt = _compute_valid_datetime(run_date, run_time, step_hours)
        slice_2d = tp.isel(step=step_idx).values if "step" in tp.dims else tp.values
        for i, lat in enumerate(lats):
            for j, lon in enumerate(lons):
                lon_norm = normalize_longitude(float(lon))
                if area is not None and not point_in_bbox(float(lat), lon_norm, area):
                    continue
                value = slice_2d[i, j]
                if value is None:
                    continue
                value_f = float(value)
                if math.isnan(value_f):
                    continue
                record = {
                    "run_date": run_date.isoformat(), "run_time": run_time, "step_hours": int(step_hours),
                    "valid_datetime": valid_dt.isoformat(), "valid_date": valid_dt.date().isoformat(),
                    "latitude": float(lat), "longitude": lon_norm, "tp_mm": value_f * unit_to_mm_factor,
                    "tipo": tipo, "source_api": source_api, "extracted_at": extracted_at,
                }
                if number is not None:
                    record["number"] = number
                records.append(record)
    return records


def write_json(records, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")


In [ ]:
try:
    dbutils.widgets.dropdown("force_reload", "false", ["false", "true"])
    force_reload = dbutils.widgets.get("force_reload").lower() == "true"
except Exception:
    force_reload = False

RAW_DIR = Path("/Volumes/weather/raw/ecmwf_volume/cf_tigge/raw")
JSON_DIR = Path("/Volumes/weather/raw/ecmwf_volume/cf_tigge/json")

DATASET = "tigge-forecasts"
ORIGIN = "ecmf"
PARAM = "228228"
STEPS_HOURS = list(range(0, 361, 24))
TIGGE_LAG_DAYS = 2
MAX_LAG_SEARCH = 5
UNIT_TO_MM_FACTOR = 1.0  # confirmado: tigge-forecasts entrega tp en kg/m2 == mm


In [ ]:
cdsapi_url = dbutils.secrets.get(scope="ecmwf", key="cdsapi_url")
cdsapi_key = dbutils.secrets.get(scope="ecmwf", key="cdsapi_key")


In [ ]:
import cdsapi
from datetime import date, timedelta

client = cdsapi.Client(url=cdsapi_url, key=cdsapi_key)
area = compute_download_area()
print(f"Area de descarga calculada: {area}")

RAW_DIR.mkdir(parents=True, exist_ok=True)


def try_retrieve(run_date, target):
    request = {
        "origin": ORIGIN, "levtype": "sfc", "param": PARAM, "type": "cf",
        "step": [str(s) for s in STEPS_HOURS], "date": run_date.isoformat(), "time": "00:00:00",
        "area": area_to_cds_list(area), "grid": [0.25, 0.25], "format": "netcdf",
    }
    try:
        client.retrieve(DATASET, request, str(target))
        return True
    except Exception as e:
        print(f"  {run_date.isoformat()} no disponible todavia: {str(e)[:150]}")
        return False


run_date = None
for lag in range(TIGGE_LAG_DAYS, MAX_LAG_SEARCH + 1):
    candidate = date.today() - timedelta(days=lag)
    if already_landed("cf", candidate, "00", JSON_DIR) and not force_reload:
        print(f"Ya existe la corrida {candidate} t00 (cf), skip")
        run_date = "SKIP"
        break
    raw_path = RAW_DIR / raw_filename("cf", candidate, "00", "nc")
    print(f"Probando corrida cf de {candidate.isoformat()} (lag={lag} dias)...")
    if try_retrieve(candidate, raw_path):
        run_date = candidate
        break

if run_date is None:
    raise SystemExit(f"No se encontro ninguna corrida cf disponible entre lag={TIGGE_LAG_DAYS} y lag={MAX_LAG_SEARCH} dias")


In [ ]:
if run_date != "SKIP":
    import xarray as xr

    ds = xr.open_dataset(RAW_DIR / raw_filename("cf", run_date, "00", "nc"), engine="netcdf4", decode_timedelta=False)
    number = int(ds["number"].values) if "number" in ds.coords else None
    records = flatten_forecast(
        ds, run_date=run_date, run_time="00", tipo="cf",
        source_api="ecmwf_tigge_cdsapi", unit_to_mm_factor=UNIT_TO_MM_FACTOR, area=None, number=number,
    )
    print(f"Registros aplanados: {len(records)}")

    json_path = JSON_DIR / raw_filename("cf", run_date, "00", "json")
    write_json(records, json_path)
    print(f"OK {json_path.name}: {len(records)} registros")
